# 02e — Combine all feature tables

Create one analysis table with **one row per `SUBJECT`/`SPELL_IDENTIFIER`**. No surrogate `episode_row_id` is used. If duplicate spell identifiers exist, this notebook keeps the first row in each input table.

In [ ]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell, keep_unique_rows

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


## Load the base episodes and derived features

In [4]:
# Load episodes using DuckDB
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes']

# Load
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = dfs['episodes']

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)


In [5]:
# 1. Load base episodes
df_base = keep_unique_rows(
    df_episodes,
    keys=["SUBJECT", "SPELL_IDENTIFIER"],
    name="base episodes",
)

# 2. Define feature files with their respective join keys
feature_sources = {
    # Spell-level features
    "phenotypes": {
        "path": DATA_PROC_DIR / "02a_derived_phenotypes_aggregated.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "history": {
        "path": DATA_PROC_DIR / "02b_derived_features_phenotypes.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "encounters": {
        "path": DATA_PROC_DIR / "02c_derived_features_encounters.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "medications": {
        "path": DATA_PROC_DIR / "02d_derived_features_medications.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "microbiology": {
        "path": DATA_PROC_DIR / "02e_derived_features_microbiology.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    # Patient-level features (broadcasts across spells)
    "demographics": {
        "path": DATA_DIR / "icare_demographic_anon.parquet",
        "keys": ["SUBJECT"]
    },
}

# 3. Validate existence and load features dynamically
loaded_features = {}
missing = []
for name, meta in feature_sources.items():
    path = meta["path"]
    if not path.exists():
        missing.append(name)
        continue

    # Read parquet or csv automatically based on extension
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    # Standardize keys to string
    for k in meta["keys"]:
        df[k] = df[k].astype(str)
    # Create dictionary
    loaded_features[name] = {
        "df": keep_unique_rows(df, meta["keys"], name),
        "keys": meta["keys"]
    }

if missing:
    raise FileNotFoundError(f"Missing feature files: {', '.join(missing)}. Ensure they are generated first.")


base episodes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
phenotypes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
history (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
encounters (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
medications (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
microbiology (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
demographics (SUBJECT): 10,000 rows -> 10,000 unique records


## Merge safely

In [7]:
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

df_combined = df_base.copy()
df_combined.SUBJECT = df_combined.SUBJECT.astype(str)
df_combined.SPELL_IDENTIFIER = df_combined.SPELL_IDENTIFIER.astype(str)

for name, item in loaded_features.items():
    feature_df = item["df"]
    keys = item["keys"]

    # Drop overlapping columns (except join keys) to avoid duplicate suffixes
    overlapping = [
        c for c in feature_df.columns
        if c in df_combined.columns and c not in KEYS
    ]
    feature_df = feature_df.drop(columns=overlapping)

    # Toggle validation rule based on table grain (spell vs patient)
    validate_rule = "one_to_one" if len(keys) > 1 else "many_to_one"

    df_combined = df_combined.merge(
        feature_df,
        on=keys,
        how="left",
        validate=validate_rule,
    )
    print(f"Merged {name} via {keys}: shape {df_combined.shape}")

assert not df_combined.duplicated(["SUBJECT", "SPELL_IDENTIFIER"]).any(), "Duplicate spells detected post-merge!"
print(f"\nFinal combined dataset ready: {len(df_combined):,} rows, {len(df_combined.columns):,} columns.")


Merged phenotypes via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 84)
Merged history via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 174)
Merged encounters via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 176)
Merged medications via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 177)
Merged microbiology via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 181)
Merged demographics via ['SUBJECT']: shape (10000, 188)

Final combined dataset ready: 10,000 rows, 188 columns.


In [8]:
filename = "03_derived_analysis_dataset.parquet"
output_path = DATA_PROC_DIR / filename
df_combined.to_parquet(output_path, index=False)
print(f"Saved {len(df_combined):,} rows and {len(df_combined.columns):,} columns to {output_path}")
display(df_combined.head())
print(df_combined.columns.tolist())


Saved 10,000 rows and 188 columns to /app/data/synthetic/10000p/processed/03_derived_analysis_dataset.parquet


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,time_to_culture_days,time_to_culture_hours,esbl_status,BIRTH_DATE,DEATH_DATE,ETHNICITY,ETHNICITY_DES,GENDER,GENDER_DESC,UPDATE_DT_TM
0,10001,7896381,9619750,5881100,2020-09-29,13:57:00,43,2020-10-07,140,oral surgery,...,NaN,NaN,unknown,1994-09-19,2020-01-07,H,Asian or Asian British - Indian,F,Female,2023-01-24
1,10002,1805970,5717880,3457211,2020-08-03,19:47:00,49,2020-08-15,320,cardiology,...,NaN,NaN,unknown,1966-09-19,2021-02-20,N,Black or Black British - African,U,Unknown,2023-04-22
2,10003,6698303,3675573,4694133,2022-11-02,16:32:00,32,2022-11-11,320,cardiology,...,NaN,NaN,unknown,1951-02-12,2024-11-05,S,Any other ethnic group,U,Unknown,2023-03-09
3,10004,6804249,7908451,5949276,2023-01-05,02:07:00,59,2023-01-10,300,general medicine,...,NaN,NaN,unknown,1960-07-14,2020-04-03,S,Any other ethnic group,M,Male,2023-12-25
4,10005,2301493,9166223,5218519,2020-11-13,19:34:00,51,2020-12-04,320,cardiology,...,NaN,NaN,unknown,2002-11-05,2024-12-04,S,Any other ethnic group,U,Unknown,2023-08-27


['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_hosp_90d', 'tumbarello_hx_recent_ab